# Analysis of screening data (LFC, FDR, p-values)

Notebook for generating log2 fold-change (LFC) values for the screen, as well as empirical p-value calculation relative to the behavior of the non-targetting guides in each library.

In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats
import os
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects
import warnings
warnings.filterwarnings('ignore')
plt.rc('font', family='Helvetica')

## Check that NT gRNA populations are represented

In [8]:
library = pd.read_csv('../../source_data/02_library/CDK_library_final.csv')
nt = library[library['classification']=='non-targeting control']
np.unique(nt['Pool'], return_counts=True)

(array(['F1-R1', 'F2-R2', 'F3-R3'], dtype=object), array([131, 235,  79]))

In [ ]:
nt_pool1 = list(nt[nt['Pool']=='F1-R1']['gRNA_id'])
nt_pool2 = list(nt[nt['Pool']=='F2-R2'])


# Empirical p-vals and FDR

Should systematically filter out NT gRNAs that have a low base mean (in whatever the denominator is in the comparison). Should do this as well for the targeting and intron-targeting gRNAs

Should only use the non-targetting guides as the null distribution (can compare results from different comparisons, but intron-targetting guides could have an effect as well)

ALSO THINK ABOUT WHEN THE PSEUDOCOUNT IS ADDED; DON'T NECESSARILY NEED TO DO IT BEFORE THE RPM CALCULATION

(updated the RPM tables such that no pseudocount is added)

https://academic.oup.com/bioinformatics/article/34/23/4054/5040306


In [ ]:
import numpy as np
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests


def one_sided_FDR_v2(condition_of_interest, LFC_df, min_DMSO_counts, num_boot=10000):
    # null distribution from NT-gRNAs (e.g., log2 fold changes)
    samp_of_interest = [f'{condition_of_interest}_REP{i}' for i in range(1,4)]

    #furthermore, add in the intron-targetting gRNAs to the distribution
    introns_NT = ['CDK19_intron', 'CDK9_intron', 'CDK7_intron', 'CDK8_intron', 'NT']

    non_targ = LFC_df[LFC_df['gene'].isin(introns_NT)].reset_index(drop=True)
    #non_targ = LFC_df[LFC_df['gene'].isin(['NT'])].reset_index(drop=True)

    non_targ = non_targ[non_targ['DMSO_median']>=min_DMSO_counts]

    data = non_targ[samp_of_interest].to_numpy().flatten()

    #nt_gRNA_values = data
    print(len(data))
    nt_gRNA_values = np.random.choice(data, size=num_boot, replace=True)
    print(len(nt_gRNA_values))

    #targeting gRNAs with 3 replicates per guide
    #including NT gRNAs here for completeness even though they're in the null distribution
    gRNA_values = np.asarray(LFC_df[samp_of_interest])

    # Function to compute empirical two-tailed p-values
    def empirical_p_value(observed, null_distribution):
        S_high = sum(null_distribution >= observed)
        S_low = sum(null_distribution <= observed)
        N = len(null_distribution)

        # Compute two-sided p-value
        p_high = (S_high + 1) / (N + 1)
        p_low = (S_low + 1) / (N + 1)
        return p_high, p_low

    # Compute p-values for each replicate separately (two-tailed)
    p_values_high = np.array([
        [empirical_p_value(rep, nt_gRNA_values)[0] for rep in gRNA] 
        for gRNA in gRNA_values
    ])

    p_values_low = np.array([
        [empirical_p_value(rep, nt_gRNA_values)[1] for rep in gRNA] 
        for gRNA in gRNA_values
    ])

    # Combine p-values across replicates using Fisher's method
    combined_p_values_high = np.array([
        combine_pvalues(p_vals, method='fisher')[1] for p_vals in p_values_high
    ])
    
    combined_p_values_low = np.array([
        combine_pvalues(p_vals, method='fisher')[1] for p_vals in p_values_low
    ])

    # Adjust for multiple testing (Benjamini-Hochberg FDR)
    adjusted_p_values_high = multipletests(combined_p_values_high, method='fdr_bh')[1]
    adjusted_p_values_low = multipletests(combined_p_values_low, method='fdr_bh')[1]

    # Store results in the dataframe
    LFC_df[f'p_high_unadjusted_{condition_of_interest}'] = combined_p_values_high
    LFC_df[f'p_low_unadjusted_{condition_of_interest}'] = combined_p_values_low
    LFC_df[f'FDR_high_{condition_of_interest}'] = adjusted_p_values_high
    LFC_df[f'FDR_low_{condition_of_interest}'] = adjusted_p_values_low
    LFC_df[f'p_unadjusted_{condition_of_interest}'] = [min(combined_p_values_high[i], combined_p_values_low[i]) for i in range(len(combined_p_values_high))]
    LFC_df[f'FDR_{condition_of_interest}'] = [min(adjusted_p_values_high[i], adjusted_p_values_low[i]) for i in range(len(adjusted_p_values_high))]

    return LFC_df #, p_values_per_replicate

In [59]:
a = os.listdir('../../screening_data/02_RPM_tables/barcode_counts')
fp = '../../screening_data/02_RPM_tables/barcode_counts'

a2 = pd.read_csv(f'{fp}/{a[4]}')
(a2 == 0).sum()

gRNA_id         0
T0_REP1         1
T0_REP2         0
T0_REP3         0
DMSO_REP1       2
DMSO_REP2       0
DMSO_REP3       1
KB_2000_REP1    2
KB_2000_REP2    0
KB_2000_REP3    3
KB_4000_REP1    2
KB_4000_REP2    1
KB_4000_REP3    2
Plasmid         0
dtype: int64

In [65]:
a = os.listdir('../../screening_data/02_RPM_tables/matched_counts')
fp = '../../screening_data/02_RPM_tables/matched_counts'

a2 = pd.read_csv(f'{fp}/{a[4]}')
(a2 == 0).sum()

gRNA_id          0
T0_REP1          9
T0_REP2          7
T0_REP3          3
DMSO_REP1        5
DMSO_REP2        5
DMSO_REP3        8
KB_2000_REP1    19
KB_2000_REP2    19
KB_2000_REP3    22
KB_4000_REP1    24
KB_4000_REP2    11
KB_4000_REP3    21
Plasmid          0
dtype: int64